# Concurrent Data Platform API Calls with Python Asyncio Gather and Data Library for Python

## What is Python Asyncio? And how it relate to Data Library for Python

Python [asyncio](https://docs.python.org/3/library/asyncio.html) is a library for writing concurrent code with `async/await`. Common asyncio interfaces for concurrent programming include:

- [`asyncio.create_task(coro)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.create_task), which wraps *one [coroutine](https://docs.python.org/3/library/asyncio-task.html#coroutine)* in a [Task](https://docs.python.org/3/library/asyncio-task.html#asyncio.Task) and schedules it for execution, returning the Task object.
- [`asyncio.gather(*aws)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather), which runs [awaitables](https://docs.python.org/3/library/asyncio-task.html#asyncio-awaitables) in the `aws` sequence concurrently.
- [Task Groups](https://docs.python.org/3/library/asyncio-task.html#task-groups), a modern task-management API that provides a reliable way to wait for all tasks in a group to finish using [`asyncio.TaskGroup`](https://docs.python.org/3/library/asyncio-task.html#asyncio.TaskGroup) together with task creation APIs such as [`asyncio.create_task(coro)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.create_task).

This notebook demonstrates how to use the [Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) Content Layer Historical Pricing `get_data_async` method for requesting multiple RICs concurrently using popular asyncio `gather` interface.

This Jupyter Notebook uses a *Platform Session* connection. If you are using another session type, please refer to the [Data Library Quickstart](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python/quick-start) for more details.

## Prerequisite  

You should have a basic understanding of Python’s asyncio before getting started. If you need a quick refresher, these resources are solid:

- [Python's asyncio: A Hands-On Walkthrough](https://realpython.com/async-io-python/)
- [Asyncio Architecture in Python: Event Loops, Tasks, and Futures Explained](https://dev.to/imsushant12/asyncio-architecture-in-python-event-loops-tasks-and-futures-explained-4pn3)
- [A Conceptual Overview of asyncio](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html)
- [Python Asynchronous I/O](https://docs.python.org/3/library/asyncio.html)

The others are the Data Platform account (Machine-ID type), Python, and Data Library for Python.

The first step is importing all requires libraries

In [2]:
import os
import asyncio
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data.content import historical_pricing
from lseg.data._errors import LDError
from lseg.data.content.historical_pricing import Adjustments, Intervals
from dotenv import load_dotenv
import pandas as pd
pd.set_option("future.no_silent_downcasting", True)

Next, use the [python-dotenv](https://pypi.org/project/python-dotenv/) library to get the Data Platform credential from the `.env` file. The `os.getenv()` method supports to OS environment variables as well if you prefer.

**Note**: The `.env` file **should not be committed** to the version control repository.

In [3]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Then, we create a library `session` object to manage an application session. The session automatic define authentication, manage connection resources, and expose the APIs to retrieve data for an application.

In [4]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

The session is opened, we can declare our historical data related variable.

In [5]:
# ── Instrument universe ────────────────────────────────────────────────────────
INSTRUMENTS = {
    "NVDA.O":  "NVIDIA",
    "AAPL.O":  "Apple",
    "MSFT.O":  "Microsoft",
    "AMZN.O":  "Amazon",
    "GOOG.O":  "Alphabet",
    "AVGO.O":  "Broadcom",
    "META.O":  "Meta",
    "ORCL.N":  "Oracle",
    "IBM.N":   "IBM",
    "PLTR.O":  "Palantir",
    "NFLX.O":  "Netflix",
    "TSLA.O":  "Tesla",
    "CRM.N":   "Salesforce",
    "AMD.O":   "AMD",
    "INTC.O":  "Intel",
    "ARM.O":   "Arm Holdings",
    "TXN.O": "Texas Instruments",
    "CSCO.O":  "Cisco Systems",
    "WMT.O":   "Walmart",
    "LLY.N":   "Eli Lilly and Company",
    "JPM.N":   "JPMorgan Chase & Co.",
    "XOM.N":   "Exxon Mobil Corporation",
    "V.N":     "Visa Inc.",
    "JNJ.N":   "Johnson & Johnson",
    "MU.O":    "Micron Technology, Inc.",
    "MA.N":    "Mastercard Incorporated",
    "COST.O":  "Costco Wholesale Corporation",
    "CVX.N":   "Chevron Corporation",
    "BAC.N":   "Bank of America Corporation",
    "CAT.N":   "Caterpillar Inc.",
}

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2025-11-01T00:00:00Z"
END   = "2026-02-28T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
# Only include exchange-level and manual corrections; filters out duplicates
# and administrative adjustments that would distort event counts.
EVENT_ADJUSTMENTS = [Adjustments.EXCHANGE_CORRECTION, Adjustments.MANUAL_CORRECTION]

# ── Field lists ─────────────────────────────────────────────────────────────────
# Defined once as constants so that each list comprehension in the next cell
# reuses the same object rather than allocating a new list on every iteration.
EVENT_FIELDS    = ["EVENT_TYPE", "TRDPRC_1", "TRDVOL_1"]
INTRADAY_FIELDS = ["TRDPRC_1", "BID", "ASK"]
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]



## Data Library Access Layer `get_history` VS Content Layer `historical_pricing`

Let's start with why we need the Data Library Content Layer Historical Pricing module instead of just the basic `get_history` method. 

The `get_history` method is part of the Data Library *Access Layer**. The Access Layer provides the easiest way to get data via a set of simple API interfaces for developers.  While the `get_history` lets developers retrieve pricing history for both single and multiple instruments via a single function call. All Access Layer methods including the `get_history` are in synchronous execution which block other tasks, an application must wait until the function call is finished. 


The `historical_pricing` module is part of the *Content Layer*. The Content Layer allows developers to access the same content as Access Layer which are a more flexible manner:
- Richer / fuller response e.g. metadata, sentiment scores - where available.
- Using Asynchronous or Event-Driven operating modes - in addition to Synchronous.
- The layer interfaces are based on logical market data objects such as Level 1 Market Price Data (snapshot/streaming), News, Historical Pricing, Bond Analytics, Environmental & Social Governance (ESG), Search, IPA, and so on.

The module lets developers set historical data query via *definition* then get data via synchronous `get_data` and asynchronous `get_data_async` methods. I am focusing on the asynchronous `get_data_async` method of the Historical Pricing module here.

## Using asyncio.gather() method with return_exceptions = True

That brings us to the most to the most direct and easiest way to request historical data concurrently, combine Historical Pricing `get_data_async` calls with [`asyncio.gather(*aws)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather) method.

**`await asyncio.gather(*aws, return_exceptions=False)`**

- Runs [awaitable objects](https://docs.python.org/3/library/asyncio-task.html#asyncio-awaitables) in the `aws` sequence concurrently.
- If all awaitables succeed, it returns a Python list of results in the same order as `aws`.
- `return_exceptions` controls how exceptions are handled:
  - If `False` (default): the first exception is raised immediately to the caller waiting on `gather()`. Other awaitables are not automatically cancelled and may continue running.
  - If `True`: exceptions are returned in the result list (instead of being raised immediately), alongside successful results.

In default mode (`return_exceptions=False`), your code may stop at the first error and not automatically collect outcomes from the other still-running awaitables. This can leave unfinished or uncollected task outcomes that are easy to miss. To handle this pattern safely, an application must keep task references and explicitly inspect task status/results when needed manually.

That is why many applications use `asyncio.gather(..., return_exceptions=True)` when they need complete visibility of both success and failure results in one place.

The first step is to define a `display_response` method to display returned historical data as a DataFrame.

In [6]:
def display_response(data):
    """Display the result of each async API call.

    For each response:
    - Prints the exception message if the task raised a Python error.
    - Warns if the response has no closure label (unexpected type).
    - Renders the DataFrame on a successful HTTP response.
    - Prints the HTTP status code on a failed (4xx/5xx) response.
    """
    for api_response in data:
        # Task raised a Python exception (e.g. network error, timeout)
        if isinstance(api_response, Exception):
            print(f"\nTask failed with exception: {api_response}")
            continue

        # Guard against unexpected response types
        if not hasattr(api_response, 'closure'):
            print(f"\nUnexpected response type: {type(api_response)}")
            continue

        print(f"\nResponse received for: {api_response.closure}")

        if api_response.is_success:
            display(api_response.data.df)
        else:
            # HTTP-level failure (4xx / 5xx from the platform)
            print(f"Request failed — HTTP status: {api_response.http_status}")



You may notice that the `display_response` method above is more defensive than the one used in [EX-2.01.02-HistoricalPricing-ParallelRequests.ipynb](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/blob/lseg-data-examples/Examples/2-Content/2.01-HistoricalPricing/EX-2.01.02-HistoricalPricing-ParallelRequests.ipynb), which only checks whether each response is successful.

```python
def display_reponse(response):
    print(response)
    print("\nReponse received for", response.closure)
    if response.is_success:
        display(response.data.df)
    else:
        print(response.http_status)
```

In this notebook, `display_response` also handles Python exceptions that can appear in the returned list when using `asyncio.gather(..., return_exceptions=True)`, in addition to HTTP-level failures. This makes concurrent request handling easier to debug and safer in real applications.

### Requesting Data with Request Limits

Next, we group multiple calls to the `get_data_async` method with `asyncio.gather()` and run them as awaitable coroutines. You can combine `asyncio.gather()` method with [`asyncio.Semaphore`](https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore) to cap the number of requests in-flight at any given time (default: 3).

If you are requesting just 2-10 RICs, the backend can handle it without issue. However, as the number of simultaneous requests grows to 30,50,100 or more, a semaphore becomes essential to stay within the platform's rate limits. The following example demonstrates this pattern with 20 RICs.

I am demonstrating with `historical_pricing.events.Definition` which returns Historical Pricing Events data similar to the Data Platform `/data/historical-pricing/v1/views/events/` endpoint.

In [7]:
# Convert dictionary keys to a list of RIC symbols (kept for quick inspection/debugging).
rics = list(INSTRUMENTS.keys())

# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_rics_companies = list(INSTRUMENTS.items())

throttle_limit = 10
semaphore = asyncio.Semaphore(throttle_limit)  # Number of simultaneous tasks to run

async def fetch_event_with_throttle(ric, company):
    """Request event data for one RIC with semaphore throttling."""
    async with semaphore:
        return await historical_pricing.events.Definition(
            universe=ric,
            fields=EVENT_FIELDS,
            count=5
        ).get_data_async(closure=company)

try:
    # Create a concurrent batch of event requests with a semaphore limit.
    tasks = [
        fetch_event_with_throttle(ric, company)
        for ric, company in list_of_rics_companies[0:20]
    ]

    historical_data = await asyncio.gather(  # pylint: disable=await-outside-async
        *tasks,
        return_exceptions=True
    )

    # Show a title for this batch.
    display(Markdown(f"**Companies Historical Price Events with Throttle {throttle_limit}**"))
    # Print each result: DataFrame on success, status/error on failure.
    display_response(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)


**Companies Historical Price Events with Throttle 10**


Response received for: NVIDIA


NVDA.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 10:36:11.818,trade,201.2832,1
2026-06-25 10:36:11.962,trade,201.24,5
2026-06-25 10:36:12.625,trade,201.29,1
2026-06-25 10:36:13.113,trade,201.29,14
2026-06-25 10:36:14.212,trade,201.29,59



Response received for: Apple


AAPL.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 10:36:07.565,trade,291.34,5
2026-06-25 10:36:07.565,trade,291.34,15
2026-06-25 10:36:12.124,trade,291.4,1
2026-06-25 10:36:13.865,trade,291.4,2
2026-06-25 10:36:14.581,trade,291.4,1


...

CSCO.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 10:34:44.194,trade,119.19,6
2026-06-25 10:34:44.194,trade,119.16,40
2026-06-25 10:35:13.110,trade,119.08,34
2026-06-25 10:35:13.868,trade,119.08,34
2026-06-25 10:36:07.566,trade,119.05,34



Response received for: Walmart


WMT.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 10:35:47.644,trade,117.42,1
2026-06-25 10:35:47.645,trade,117.4,1
2026-06-25 10:36:07.565,trade,117.42,1
2026-06-25 10:36:07.565,trade,117.41,3
2026-06-25 10:36:08.452,trade,117.42,1



Response received for: Eli Lilly and Company


LLY.N,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-24 20:01:44.259,trade,1117.26,632161
2026-06-24 20:01:44.367,trade,1117.26,<NA>
2026-06-24 20:10:00.003,trade,1117.26,<NA>
2026-06-24 22:30:00.003,trade,1117.26,<NA>
2026-06-24 23:00:00.003,trade,1117.26,<NA>


Please be noticed that when developers send multiple Historical Pricing Definition with **a single RIC** request, each RIC gets its own data response grouping together sequently in a Python *list* returns from `await tasks` statement.

In [8]:
print(f" Data type is {type(historical_data)} and length is {len(historical_data)}")

 Data type is <class 'list'> and length is 20


And application can iterate each result from the returned list (as shown in the `display_response` method) or use the following statement to get data from its closure.

In [9]:
next(
    response.data.df
    for response in historical_data
    if getattr(response, "closure", None) == "NVIDIA"
 )

NVDA.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 10:36:11.818,trade,201.2832,1
2026-06-25 10:36:11.962,trade,201.24,5
2026-06-25 10:36:12.625,trade,201.29,1
2026-06-25 10:36:13.113,trade,201.29,14
2026-06-25 10:36:14.212,trade,201.29,59


### Understanding asyncio.gather with return_exceptions=True

This section combines request throttling (`asyncio.Semaphore`) with `asyncio.gather(..., return_exceptions=True)` so the full batch completes even if some requests fail.

#### Why use return_exceptions=True

With `return_exceptions=True`, `asyncio.gather` returns one list that may contain both:

- successful response objects
- exception objects for failed tasks

This behavior is useful for batch workflows because one failed request does not stop the remaining requests.

#### How to read the returned data safely

After:

```python
historical_data = await asyncio.gather(*tasks, return_exceptions=True)
```

iterate through each item and handle by type:

1. If the item is an `Exception`, log or report it.
2. If it is a successful API response (`response.is_success`), read data from `response.data.df`.
3. If the API response is not successful, inspect `response.http_status`.

The helper `display_response(historical_data)` method in this notebook already follows this defensive pattern.

#### How to get data for one instrument

Each request uses `closure=company`, so you can retrieve one instrument by matching `closure`:

```python
next(
    response.data.df
    for response in historical_data
    if getattr(response, "closure", None) == "NVIDIA"
)
```

#### Where Semaphore fits

When sending a large number of concurrent requests, an `asyncio.Semaphore` is **required** to avoid exceeding the platform's backend rate limit. Without it, all coroutines are submitted to the event loop simultaneously, and the platform will respond with HTTP **429 Too Many Requests** errors.

##### How it works

A semaphore is a counter that allows at most *N* coroutines to hold it at the same time. Any coroutine that attempts to acquire the semaphore when the counter is exhausted will suspend and wait until another coroutine releases it.

```
asyncio.Semaphore(N)  →  at most N HTTP requests in-flight at any time
```

##### Key points

- **Define the semaphore once** outside the helper function and close over it — do not create a new instance per call.
- **Wrap `await get_data_async(...)`** inside `async with semaphore:` — this is the only critical section that needs throttling.
- **The semaphore controls concurrency inside each coroutine**, not inside `asyncio.gather` itself. Pass the coroutine list to `gather` as usual.
- **Tune N to your account tier.** A value between 5 and 10 is a safe starting point for most Data Platform accounts. Reduce it further if you continue to see 429 responses.

##### Pattern recap

```python
semaphore = asyncio.Semaphore(10)   # cap: at most 10 simultaneous in-flight requests

async def fetch_with_throttle(ric, company):
    async with semaphore:            # suspends here when 10 requests are already in-flight
        return await historical_pricing.events.Definition(
            universe=ric, fields=EVENT_FIELDS, count=5
        ).get_data_async(closure=company)

historical_data = await asyncio.gather(
    *[fetch_with_throttle(ric, company) for ric, company in list_of_rics_companies],
    return_exceptions=True
)
```

Using both together gives controlled concurrency and complete result visibility.

### What about Historical Pricing Summaries Definition?

You can use the `asyncio.gather` with `historical_pricing.summaries.Definition` definition which is similar from Data Platform `/data/historical-pricing/v1/views/intraday-summaries/` endpoint

In [10]:
throttle_limit = 10
semaphore = asyncio.Semaphore(throttle_limit)  # Number of simultaneous tasks to run


async def fetch_intraday_with_throttle(ric, company):
    """Fetch intraday data for a given RIC and company with concurrency limit."""
    async with semaphore:
        return await historical_pricing.summaries.Definition(
            universe=ric,
            fields=INTRADAY_FIELDS,
            count=5,
            interval=Intervals.FIVE_MINUTES,
        ).get_data_async(closure=company)


try:
    # Create a concurrent batch of intraday requests with a semaphore limit.
    tasks = [
        fetch_intraday_with_throttle(ric, company)
        for ric, company in list_of_rics_companies[10:30]
    ]

    historical_data = await asyncio.gather(  # pylint: disable=await-outside-async
        *tasks,
        return_exceptions=True
    )

    # Show a title for this batch.
    display(
        Markdown(
            "**Companies Historical Price Intraday data (5-minute intervals) "
            f"with Throttle {throttle_limit}**"
        )
    )
    # Print each result: DataFrame on success, status/error on failure.
    display_response(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Intraday data (5-minute intervals) with Throttle 10**


Response received for: Netflix


NFLX.O,TRDPRC_1,BID,ASK
Timestamp,,,
2026-06-25 10:15:00,71.57,71.54,71.6
2026-06-25 10:20:00,71.5022,71.5,71.59
2026-06-25 10:25:00,71.57,71.5,71.57
2026-06-25 10:30:00,71.53,71.5,71.55
2026-06-25 10:35:00,71.57,71.5,71.62



Response received for: Tesla


TSLA.O,TRDPRC_1,BID,ASK
Timestamp,,,
2026-06-25 10:15:00,378.49,378.33,378.64
2026-06-25 10:20:00,378.4,378.3,378.6
2026-06-25 10:25:00,378.2232,378.22,378.35
2026-06-25 10:30:00,378.31,378.03,378.46
2026-06-25 10:35:00,378.26,378.26,378.46


...

CVX.N,TRDPRC_1,BID,ASK
Timestamp,,,
2026-06-24 19:55:00,171.56,171.5,171.55
2026-06-24 20:00:00,171.45,171.55,171.58
2026-06-24 20:10:00,171.45,<NA>,<NA>
2026-06-24 22:30:00,171.45,<NA>,<NA>
2026-06-24 23:00:00,171.45,<NA>,<NA>



Response received for: Bank of America Corporation


BAC.N,TRDPRC_1,BID,ASK
Timestamp,,,
2026-06-24 19:55:00,57.745,57.74,57.75
2026-06-24 20:00:00,57.73,0.0,0.0
2026-06-24 20:10:00,57.73,<NA>,<NA>
2026-06-24 22:30:00,57.73,<NA>,<NA>
2026-06-24 23:00:00,57.73,<NA>,<NA>



Response received for: Caterpillar Inc.


CAT.N,TRDPRC_1,BID,ASK
Timestamp,,,
2026-06-24 19:55:00,994.74,994.41,994.74
2026-06-24 20:00:00,994.45,<NA>,<NA>
2026-06-24 20:10:00,994.45,<NA>,<NA>
2026-06-24 22:30:00,994.45,<NA>,<NA>
2026-06-24 23:00:00,994.45,<NA>,<NA>


### How the return_exceptions = True handle errors?

When using `asyncio.gather` with `return_exceptions=True`, the errors and exceptions are returns in the result list along side the success ones. 

**Note**: This error-handling example uses only a small number of RICs, so `asyncio.Semaphore` is not required in this section.

#### Invalid and Non-Permission RICs Request

I am demonstrating with the invalid RIC code `INVALID_RIC` and non-permission RIC (`ASML.L` for ASML Holding, your permission may be different) requests.

In [11]:
invalid_instrument_cases = {"IBM.N":"IBM","INVALIDRIC.O": "Invalid Instrument","JPM.N":   "JPMorgan Chase & Co.","ASML.AS": "ASML"}

# Convert dictionary keys to a list of RIC symbols (kept for quick inspection/debugging).
rics_with_errors = list(invalid_instrument_cases.keys())

# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_rics_companies_with_errors = list(invalid_instrument_cases.items())

try:
    # Create a concurrent batch of event requests for the first three instruments.
    # The star-unpack passes each coroutine as a separate argument to gather.
    tasks = asyncio.gather(
        *[
            historical_pricing.summaries.Definition(universe=ric, fields=INTERDAY_FIELDS, count=5, interval = Intervals.DAILY).get_data_async(closure=company)
            for ric, company in list_of_rics_companies_with_errors
        ],
        return_exceptions=True  # Prevent gather from raising immediately on the first exception; we want to collect all results.
    )

    # Wait for the entire batch to finish and collect all response objects.
    # Default gather behavior: if any task raises an exception, it is raised at this await line.
    historical_data = await tasks  # pylint: disable=await-outside-async

    # Display a section header before printing each response output.
    display(Markdown("**Companies Historical Price Summaries with RIC error**"))
    # Show each response DataFrame on success; otherwise print the HTTP status code.
    display_response(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Summaries with RIC error**


Response received for: IBM


IBM.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-06-17,262.26,262.35,266.0,268.85,261.89,262.35,15817,422556450
2026-06-18,249.18,249.21,249.61,252.4,243.74,249.1,26814,1569756478
2026-06-22,252.45,252.52,248.8,253.31,243.86,252.22,20899,649808876
2026-06-23,264.81,264.95,261.58,267.5,255.59,264.94,32670,718822849
2026-06-24,262.92,263.04,261.04,265.06,256.46,262.96,16736,531866379



Task failed with exception: Error code TS.Interday.UserRequestError.70005 | The universe is not found.. Requested ric: INVALIDRIC.O. Requested fields: ['BID', 'ASK', 'OPEN_PRC', 'HIGH_1', 'LOW_1', 'TRDPRC_1', 'NUM_MOVES', 'TRNOVR_UNS']

Response received for: JPMorgan Chase & Co.


JPM.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-06-17,333.47,333.57,332.21,337.76,331.63,333.46,34065,1095946481
2026-06-18,325.23,325.24,336.69,338.06,324.165,325.22,31181,3432143880
2026-06-22,331.76,332.04,329.0,332.77,326.815,331.48,26818,1098000762
2026-06-23,334.14,334.22,329.51,335.35,327.22,334.14,24863,890286736
2026-06-24,333.72,333.73,333.54,334.53,329.86,333.45,21112,1095717537



Task failed with exception: Error code TSCC.QS.UserNotPermission.92000 | User has no permission.. Requested ric: ASML.AS. Requested fields: ['BID', 'ASK', 'OPEN_PRC', 'HIGH_1', 'LOW_1', 'TRDPRC_1', 'NUM_MOVES', 'TRNOVR_UNS']



You can see that the results include both successful responses and error messages:

- `INVALIDRIC.O` returns `The universe is not found.. Requested ric: INVALIDRIC.O` message, which means the instrument was not found.
- `ASML.AS` returns `User has no permission.. Requested ric: ASML.AS` message, which means the user does not have permission to access that instrument.

These error messages appear alongside the historical data returned for the successful requests.

#### Invalid Fields

Now let's see how the library handles invalid fields with the `asyncio.gather`.

In [12]:
EVENT_FIELDS_WITH_INVALID = EVENT_FIELDS + ["INVALID_FIELD"]  # Invalid field to demonstrate error handling
try:
    # Create a concurrent batch of event requests for the first three instruments.
    # The star-unpack passes each coroutine as a separate argument to gather.
    tasks = asyncio.gather(
        historical_pricing.events.Definition(universe="VOD.L", fields=EVENT_FIELDS_WITH_INVALID, count=5).get_data_async(closure="Vodaphone"),
        return_exceptions=True  # Prevent gather from raising immediately on the first exception; we want to collect all results.
    )

    # Wait for the entire batch to finish and collect all response objects.
    # Default gather behavior: if any task raises an exception, it is raised at this await line.
    historical_data = await tasks  # pylint: disable=await-outside-async

    # Display a section header before printing each response output.
    display(Markdown("**Companies Historical Price Events with Field(s) error**"))
    # Show each response DataFrame on success; otherwise print the HTTP status code.
    display_response(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Events with Field(s) error**


Response received for: Vodaphone


VOD.L,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-06-25 09:14:48.462,trade,105.1,4.376784
2026-06-25 09:15:00.313,trade,105.1,5.918173
2026-06-25 09:15:00.313,trade,105.05,28.0
2026-06-25 09:15:51.177,trade,105.075,842.0
2026-06-25 09:15:55.017,trade,105.05,2671.0


The library can handle a mix of valid and invalid fields in the same request. The `response.data.df` statement output omits invalid fields automatically.

You can inspect the field-level errors in the raw response via `response.data.raw` statement

In [13]:
historical_data[0].data.raw

{'universe': {'ric': 'VOD.L'},
 'adjustments': ['exchangeCorrection', 'manualCorrection'],
 'defaultPricingField': 'TRDPRC_1',
 'qos': {'timeliness': 'delayed'},
 'headers': [{'name': 'DATE_TIME', 'type': 'string'},
  {'name': 'EVENT_TYPE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRDVOL_1', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25T09:15:55.017000000Z', 'trade', 105.05, 2671],
  ['2026-06-25T09:15:51.177000000Z', 'trade', 105.075, 842],
  ['2026-06-25T09:15:00.313000000Z', 'trade', 105.1, 5.91817316],
  ['2026-06-25T09:15:00.313000000Z', 'trade', 105.05, 28],
  ['2026-06-25T09:14:48.462000000Z', 'trade', 105.1, 4.37678401]],
 'status': {'code': 'TS.Intraday.UserRequestError.90006',
  'message': 'The universe does not support the following fields: [INVALID_FIELD].'},
 'meta': {'blendingEntry': {'headers': [{'name': 'COLLECT_DATETIME',
     'type': 'string'},
    {'name': 'RTL', 'type': 'number', 'decimalChar': '.'

You see that the error is available in raw data result from the platform. You can use the raw information to inform users if you need.

```json
{'code': 'TS.Intraday.UserRequestError.90006',
  'message': 'The universe does not support the following fields: [INVALID_FIELD].'}
```

Please note that if you send a request with only invalid fields (either one invalid field or a list of all invalid fields), the request fails and returns an error to the application.

In [14]:
try:
    # Create a concurrent batch of event requests for the first three instruments.
    # The star-unpack passes each coroutine as a separate argument to gather.
    tasks = asyncio.gather(
        historical_pricing.events.Definition(universe="VOD.L", fields="INVALID_FIELD", count=5).get_data_async(closure="Vodaphone"),
        return_exceptions=True  # Prevent gather from raising immediately on the first exception; we want to collect all results.
    )

    # Wait for the entire batch to finish and collect all response objects.
    # Default gather behavior: if any task raises an exception, it is raised at this await line.
    historical_data = await tasks  # pylint: disable=await-outside-async

    # Display a section header before printing each response output.
    display(Markdown("**Companies Historical Price Events with single-Field request error**"))
    # Show each response DataFrame on success; otherwise print the HTTP status code.
    display_response(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Events with single-Field request error**


Task failed with exception: Error code TS.Intraday.UserRequestError.90006 | The universe does not support the following fields: [INVALID_FIELD]. Requested ric: VOD.L


### Can I use Historical Pricing Events and Summaries in the same asyncio.gather

Off cause, you can. Please see an example (with `return_exception=False`) in the [Content layer - How to send parallel requests](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/blob/lseg-data-examples/Examples/2-Content/2.01-HistoricalPricing/EX-2.01.02-HistoricalPricing-ParallelRequests.ipynb) example on GitHup repository.

That is all I want to say about the Data Library Historical Pricing with Asyncio Gather method.

Now we come to the last section of the code, you can close the session with the following statements.

In [24]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()

## What about the list of RICs?

The Historical Pricing definitions universe parameter accept both single-RIC and list-of-RICs inputs.

**Single-RIC approach** (recommended): Each request returns its own dataframe and raw json response, making it easy to handle successes and failures individually.

**List-of-RICs approach**: A single request returns a [multi-index](https://pandas.pydata.org/docs/user_guide/advanced.html#multiindex-advanced-indexing) dataframe with data from all RICs combined along with an array of JSON data. This is harder to manage and parse errors per individual instrument.

**Recommendation**: Use multiple single-RIC requests with `asyncio.gather()` for better data handling, as each instrument’s success or failure can be handled independently.

## Using Data Library Historical Pricing with Asyncio Gather Summary

`asyncio.gather(..., return_exceptions=True)` is practical for concurrent batch requests when you need full visibility of all outcomes (success and fail).

### What it does

- Runs all request coroutines concurrently.
- Returns one result list in the same order as the input coroutines.
- Keeps successful responses and exceptions together in that list, instead of failing immediately on the first error.

### Why this is useful

- You can still process valid instruments even when some requests fail.
- Error handling is simpler for batch workflows because all outcomes are collected in one place.
- It is easier to build clear logs and user-friendly reports from a single result list.

### How to read the results safely

- Check each item in the returned list.
- If the item is an exception, record or print the error message.
- If the item is a successful response, process `response.data.df` as usual.

### Good use cases

- Best-effort batch requests across many RICs.
- Monitoring jobs where partial data is still valuable.
- Exploratory workflows where you want both data and errors in one run.

### Throttle Requests

- Always use `asyncio.Semaphore` to control how many requests are in-flight at the same time.
- Place the semaphore inside each request coroutine (`async with semaphore:`), not around `asyncio.gather(...)`.
- Start with a conservative limit (for example, 5-10), then tune based on your account limits and observed behavior.
- If you see HTTP 429 (Too Many Requests), reduce the semaphore limit and retry.

### Performance

For a performance comparison, refer to the [Historical Pricing get_data_async with Asyncio.Gather Performance](/notebook/ld_notebook_gather_performance.ipynb) and [Data Library Get History Synchronous Performance](/notebook/ld_notebook_gethistory_performance.ipynb) examples, both of which retrieve interday historical data for 30 instruments.

Please note that both examples measure retrieval time only, excluding display overhead.

### Important note

The `return_exceptions=True` option does not hide errors. It returns errors as list items, so your code must explicitly handle both successes and exceptions.

## Is `asyncio.gather` the only way to run concurrent tasks in Python?

No. `asyncio.gather()` is widely used, but it is not the only option for running concurrent tasks.

Depending on your application requirements, you can also use:
- `asyncio.create_task(...)` + explicit `await`: start tasks immediately and await them when appropriate.
- `asyncio.as_completed(...)`: process results as each task finishes.
- `asyncio.wait(...)`: apply lower-level coordination, such as timeouts or partial completion.
- `asyncio.to_thread(...)` / executors: move blocking I/O or CPU-intensive work outside the event loop.
- `asyncio.TaskGroup` (Python 3.11+): use structured concurrency with safer and clearer task lifecycle management.

Among these approaches, `TaskGroup` is now a common choice and is frequently compared with `gather` in modern asyncio design discussions for safer task lifecycle management.

### What Next?

Please wait for how to use Data Library Historical Pricing `get_data_async` with `asyncio.TaskGroup` in the next example.

## Should I use Data Library or the manual HTTP REST API Coding?

Before I finish, there is one point lef, should you use the Data Library or the manual HTTP REST coding? 

If you are using [Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python), [C#/.NET](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-net), or [TypeScript](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-typescript), the Data Library offers the following advantages over working directly with the HTTP REST APIs:

1. The Library automatically manages Data Platform authentication and sessions for you, so you do not need to handle sign-in, session expiration, or access-token refresh manually.
2. The Library provides developer-friendly interfaces for sending HTTP data requests. These interfaces range from simple one-line methods in the Access Layer, to richer methods in the Content Layer for more advanced use cases, to lower-level Delivery Layer methods that let you control headers, URLs, parameters, and request bodies while still handling authentication for the application.

However, if you prefer to manage authentication and sessions yourself, or if you are using another programming language such as Java, Go, Rust, Ruby, or C++, the Data Platform HTTP REST APIs are also straightforward and easy to use.

That covers all I wanted to say today. 